# The Black-Swan Test: does a small frontier LLM treat a counterexample as a falsification?

**Model:** `Qwen/Qwen3-4B` (hybrid thinking / non-thinking, 36 layers). Runs on a free Colab T4.

**Setup.** Each prompt is a few-shot classification puzzle. Every line shows two features and a label:

```
wex quv -> 2
jom zol -> 1
...
fip quv ->        <- query: the two features disagree
```

One feature (the **rule**) predicts the label on *every* demo. The other (the **shortcut**) predicts it on
every demo *except* `m` counterexamples. The query is a **conflict**: the rule says one label, the shortcut says
another. We read the model's next-token probabilities over the labels and report

`shortcut_pref = p(shortcut label) / (p(shortcut label) + p(rule label))`.

**What a scientist would do.** With `m = 0` the demos cannot tell the two features apart, so 0.5 is the
*correct* answer. With `m >= 1` the shortcut has been falsified and the rule never has. An ideal
Bayesian observer drops toward 0, and its answer depends only on the **count** `m`, not on how many confirming
demos surround it (one black swan refutes "all swans are white" whether you saw 10 white swans or 1000).

**What standard benchmarks measure.** Accuracy on queries where the two features *agree*, so both readings
score 100% and the failure is invisible.

Experiments:
1. Black-swan curve: `shortcut_pref` vs `m` across three prompt tiers, against an ideal observer.
2. Count vs fraction: hold `m = 1`, add more confirming demos. Bayes predicts flat; vote-counting predicts the counterexample gets diluted.
3. Internals: does the residual stream *know* which feature is the rule (linear probe, logit lens) even when the output does not act on it?
4. Causal test: steer along the model's own "which feature is the rule" direction (held-out, with a random-direction control).
5. (Optional) Does thinking mode fix it?

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate "huggingface_hub>=1.0" scikit-learn matplotlib

In [ ]:
import os, json, math, time, random
import numpy as np
import torch
import matplotlib.pyplot as plt

MODEL_ID = "Qwen/Qwen3-4B"      # swap to "Qwen/Qwen3-1.7B" for a faster run
QUICK = False                   # True = small N smoke test (~5 min)
N = 40 if QUICK else 200        # tasks per condition (each task is an independent unit)
BATCH = 16
SEED = 0
OUT = "fatima_out"
os.makedirs(f"{OUT}/figures", exist_ok=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 1. The task generator (exact control over the number of counterexamples `m`)

In [ ]:
POOLS = {
    # pronounceable nonsense: no lexical prior about which feature "should" matter
    "nonsense": (["wex", "jom", "fip", "vab", "kir", "dov", "plim", "greb", "tannu", "yolp"],
                 ["quv", "zol", "bik", "nel", "tor", "suan", "mig", "rax", "poth", "lend"]),
    # everyday words: object + colour
    "natural": (["cup", "key", "lamp", "coin", "book", "shoe", "drum", "kite", "bell", "ring"],
                ["red", "blue", "green", "yellow", "purple", "orange", "pink", "brown", "gray", "white"]),
}
HINT = ("Each line lists two features and a label. Exactly one of the two features determines the label "
        "on every line; the other only usually matches. Work out which feature is the rule and apply it.\n\n")
TIERS = ["nonsense", "natural", "hint"]       # hint = nonsense pools + explicit instruction
K = 3                                          # number of classes / labels
LABELS = [str(i + 1) for i in range(K)]


def make_task(rng, tier="nonsense", k_per=4, m=0, conflict=True):
    """One few-shot task. `m` demos (from distinct classes) get their shortcut word swapped to another
    class's shortcut word, so the shortcut is falsified m times while staying the majority cue."""
    assert 0 <= m <= K and k_per >= 2
    pool = POOLS["nonsense" if tier == "hint" else tier]
    flip = rng.random() < 0.5                       # which pool plays the rule (counterbalanced)
    rule_pool, short_pool = (pool[1], pool[0]) if flip else (pool[0], pool[1])
    rule_first = bool(rng.random() < 0.5)           # which slot the rule occupies (counterbalanced)
    rw = list(rng.choice(rule_pool, K, replace=False))
    sw = list(rng.choice(short_pool, K, replace=False))
    lab = [LABELS[i] for i in rng.permutation(K)]   # class c -> label lab[c]

    demos = [[c, rw[c], sw[c], lab[c]] for c in range(K) for _ in range(k_per)]
    flipped_classes = rng.choice(K, m, replace=False)
    for c in flipped_classes:                       # counterexample: shortcut word from another class
        idx = [i for i, d in enumerate(demos) if d[0] == c and d[2] == sw[c]][0]
        other = int(rng.choice([o for o in range(K) if o != c]))
        demos[idx][2] = sw[other]
    rng.shuffle(demos)

    # query: rule class a, shortcut class b (b != a for a conflict). The exact query pair must NOT appear
    # in the demos -- otherwise a counterexample can be the query itself and the model just looks it up.
    seen = {(r, s) for _, r, s, _ in demos}
    pairs = [(a, b) for a in range(K) for b in range(K)
             if ((b != a and (rw[a], sw[b]) not in seen) if conflict else b == a)]
    a, b = pairs[int(rng.integers(len(pairs)))]
    line = (lambda r, s: f"{r} {s}") if rule_first else (lambda r, s: f"{s} {r}")
    body = "\n".join(f"{line(r, s)} -> {l}" for _, r, s, l in demos) + f"\n{line(rw[a], sw[b])} ->"
    prompt = (HINT if tier == "hint" else "") + body
    # local evidence: was the query's own shortcut word ever shown with a label other than its majority label?
    word_falsified = bool(any(s == sw[b] and l != lab[b] for _, _, s, l in demos))
    return dict(prompt=prompt, tier=tier, k_per=k_per, m=m, conflict=conflict,
                rule_label=lab[a], short_label=lab[b], rule_first=rule_first,
                rule_pool=("B" if flip else "A"), query_word_falsified=word_falsified, n_demos=K * k_per)


def ideal_pref(m, eps):
    """Bayes-optimal shortcut_pref for an observer who believes each demo label is noise with prob eps.
    Two hypotheses (rule-feature vs shortcut-feature determines the label), equal priors.
    Depends on m only (the count), never on the number of confirming demos."""
    if eps <= 0:
        return 0.5 if m == 0 else 0.0
    r = (eps / ((K - 1) * (1 - eps))) ** m          # likelihood ratio shortcut : rule
    p_s = r / (1 + r)
    p_short = p_s * (1 - eps) + (1 - p_s) * eps / (K - 1)
    p_rule = (1 - p_s) * (1 - eps) + p_s * eps / (K - 1)
    return p_short / (p_short + p_rule)


def implied_eps(pref, m):
    """Invert ideal_pref: the label-noise belief that would make a Bayesian match the model."""
    if m == 0:
        return float("nan")
    lo, hi = 1e-6, (K - 1) / K - 1e-6               # at eps=(K-1)/K labels are pure noise -> pref 0.5
    if pref <= ideal_pref(m, lo): return lo
    if pref >= ideal_pref(m, hi): return hi
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if ideal_pref(m, mid) < pref else (lo, mid)
    return (lo + hi) / 2


rng = np.random.default_rng(1)
for tier, m in [("nonsense", 0), ("nonsense", 2), ("natural", 1)]:
    t = make_task(rng, tier=tier, m=m)
    print(f"--- tier={tier} m={m}  rule_label={t['rule_label']} shortcut_label={t['short_label']} "
          f"rule_first={t['rule_first']}\n{t['prompt']}\n")
print({m: round(ideal_pref(m, 0.05), 3) for m in range(4)}, "<- ideal observer, eps=0.05")

## 2. Build and save the evaluation dataset

In [ ]:
rng = np.random.default_rng(SEED)
DATA = []
def add(split, **kw):
    t = make_task(rng, **kw); t["split"] = split; t["id"] = len(DATA); DATA.append(t)

for tier in TIERS:                                       # E1 black-swan curve
    for m in range(K + 1):
        for _ in range(N): add("E1_blackswan", tier=tier, k_per=4, m=m)
    for _ in range(N): add("E0_aligned_sanity", tier=tier, k_per=4, m=1, conflict=False)
for k_per in [2, 3, 4, 6]:                               # E2 count vs fraction (m = 1 fixed)
    for _ in range(N): add("E2_count_vs_fraction", tier="nonsense", k_per=k_per, m=1)
for m in [0, 1, 2]:                                      # E3 internals: probe / logit lens / steering
    for _ in range(2 * N): add("E3_internals", tier="nonsense", k_per=4, m=m)

def query_in_demos(t):
    lines = t["prompt"].strip().split("\n")
    return any(l.split(" -> ")[0] == lines[-1][:-3] for l in lines[:-1] if " -> " in l)
assert not any(query_in_demos(t) for t in DATA if t["conflict"]), "query pair leaked into demos"

with open(f"{OUT}/blackswan_eval.jsonl", "w") as f:
    for t in DATA: f.write(json.dumps(t) + "\n")
print(len(DATA), "tasks saved")

## 3. Load the model and check label tokenisation

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
dtype = torch.bfloat16 if (DEV == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if DEV == "cuda" else torch.float32)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(DEV).eval()
NL = model.config.num_hidden_layers
print(MODEL_ID, "dtype", dtype, "layers", NL)

# " 1" tokenises as [space, digit] in Qwen; we append the space and read the digit as the next token.
sp = {L: tok(f" {L}", add_special_tokens=False)["input_ids"] for L in LABELS}
assert all(len(v) == 2 and v[0] == sp[LABELS[0]][0] for v in sp.values()), sp
SPACE_ID = sp[LABELS[0]][0]
LAB_ID = torch.tensor([sp[L][1] for L in LABELS], device=DEV)
print("space id", SPACE_ID, "label ids", LAB_ID.tolist())

try:
    from huggingface_hub import model_info
    MODEL_SHA = model_info(MODEL_ID).sha
except Exception:
    MODEL_SHA = "unknown"

### Scoring: one batched forward pass per task gives label probabilities, plus (optionally)
the last-position residual stream at every layer and a logit-lens readout per layer.

In [ ]:
def _encode(prompts):
    ids = [tok(p, add_special_tokens=False)["input_ids"] + [SPACE_ID] for p in prompts]
    T = max(map(len, ids))
    inp = torch.full((len(ids), T), tok.pad_token_id); att = torch.zeros((len(ids), T), dtype=torch.long)
    for i, x in enumerate(ids):
        inp[i, T - len(x):] = torch.tensor(x); att[i, T - len(x):] = 1
    pos = (att.cumsum(1) - 1).clamp(min=0)
    return inp.to(DEV), att.to(DEV), pos.to(DEV)


@torch.no_grad()
def score(tasks, hidden=False, lens=False, hook=None):
    """Returns per-task label probs (n,K); optionally last-token residuals (n, NL+1, d) and
    logit-lens label probs per layer (n, NL+1, K). `hook` = (layer_idx, fn(h_last, batch_tasks)->h_last)."""
    P, H, LL = [], [], []
    for s in range(0, len(tasks), BATCH):
        bt = tasks[s:s + BATCH]
        inp, att, pos = _encode([t["prompt"] for t in bt])
        hnd = None
        if hook is not None:
            li, fn = hook
            def _h(mod, i, out, bt=bt):
                h = out[0] if isinstance(out, tuple) else out
                h = h.clone(); h[:, -1, :] = fn(h[:, -1, :], bt)
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            hnd = model.model.layers[li - 1].register_forward_hook(_h)   # output of block li = hidden_states[li]
        try:
            o = model(input_ids=inp, attention_mask=att, position_ids=pos, output_hidden_states=hidden or lens)
        finally:
            if hnd: hnd.remove()
        P.append(torch.softmax(o.logits[:, -1, LAB_ID].float(), -1).cpu())
        if hidden or lens:
            hs = torch.stack([h[:, -1, :] for h in o.hidden_states], 1)      # (b, NL+1, d)
            if hidden: H.append(hs.float().cpu())
            if lens:
                normed = torch.cat([model.model.norm(hs[:, :-1]), hs[:, -1:]], 1)  # last entry is already normed
                LL.append(torch.softmax(model.lm_head(normed)[..., LAB_ID].float(), -1).cpu())
    out = {"p": torch.cat(P).numpy()}
    if hidden: out["H"] = torch.cat(H).numpy()
    if lens: out["lens"] = torch.cat(LL).numpy()
    return out


def prefs(tasks, p):
    i_s = np.array([LABELS.index(t["short_label"]) for t in tasks])
    i_r = np.array([LABELS.index(t["rule_label"]) for t in tasks])
    ps, pr = p[np.arange(len(tasks)), i_s], p[np.arange(len(tasks)), i_r]
    return ps / (ps + pr + 1e-12)


def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x); r = np.random.default_rng(seed)
    b = r.choice(x, (n, len(x))).mean(1)
    return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


by = lambda **kw: [t for t in DATA if all(t[k] == v for k, v in kw.items())]
RES = {"model": MODEL_ID, "model_sha": MODEL_SHA, "N_per_cell": N, "seed": SEED}

### Sanity: on *aligned* queries (both features agree) the model should be near-perfect.
This is the number a standard benchmark would report.

In [ ]:
RES["E0_aligned_acc"] = {}
for tier in TIERS:
    ts = by(split="E0_aligned_sanity", tier=tier)
    p = score(ts)["p"]
    acc = np.mean([LABELS[i] == t["rule_label"] for i, t in zip(p.argmax(1), ts)])
    RES["E0_aligned_acc"][tier] = float(acc)
    print(f"aligned accuracy [{tier}]: {acc:.3f}")

## Experiment 1: the black-swan curve

In [ ]:
t0 = time.time()
E1 = {}
for tier in TIERS:
    for m in range(K + 1):
        ts = by(split="E1_blackswan", tier=tier, m=m)
        p = score(ts)["p"]; pr = prefs(ts, p)
        for t, x, pp in zip(ts, pr, p): t["shortcut_pref"] = float(x); t["label_probs"] = pp.tolist()
        mu, lo, hi = boot_ci(pr)
        hard = float(np.mean(pr > 0.5))
        E1[f"{tier}|m{m}"] = dict(mean=mu, lo=lo, hi=hi, hard_shortcut_rate=hard, implied_eps=implied_eps(mu, m))
        print(f"{tier:9s} m={m}: shortcut_pref={mu:.3f} [{lo:.3f},{hi:.3f}]  argmax->shortcut {hard:.2f}  "
              f"ideal(eps=.05)={ideal_pref(m, .05):.3f}  implied eps={E1[f'{tier}|m{m}']['implied_eps']:.3f}")
RES["E1_blackswan"] = E1
print(f"{time.time() - t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ms = np.arange(K + 1)
for tier, c in zip(TIERS, ["C0", "C1", "C2"]):
    y = [E1[f"{tier}|m{m}"]["mean"] for m in ms]
    e = np.array([[y[i] - E1[f"{tier}|m{m}"]["lo"], E1[f"{tier}|m{m}"]["hi"] - y[i]] for i, m in enumerate(ms)]).T
    ax.errorbar(ms, y, yerr=e, marker="o", capsize=3, color=c, label=f"{MODEL_ID.split('/')[-1]} · {tier}")
for eps, ls in [(0.0, "-"), (0.05, "--"), (0.2, ":")]:
    ax.plot(ms, [ideal_pref(m, eps) for m in ms], "k", ls=ls, lw=1, label=f"ideal observer, ε={eps}")
ax.axhline(0.5, color="gray", lw=0.5)
ax.set_xlabel("m = counterexamples to the shortcut in the demos"); ax.set_ylabel("shortcut preference")
ax.set_xticks(ms); ax.set_ylim(-0.02, 0.7); ax.legend(fontsize=8); ax.set_title("Black-swan curve")
fig.tight_layout(); fig.savefig(f"{OUT}/figures/e1_blackswan_curve.png", dpi=160); plt.show()

**Local vs global falsification.** A falsified rule is falsified everywhere. Does the model only discount the
shortcut when the *query's own* shortcut word was shown with a wrong label? (Stratified by m, since the
two groups have different m mixes.)

In [ ]:
RES["E1_local_vs_global"] = {}
for tier in TIERS:
    for m in range(1, K + 1):
        ts = by(split="E1_blackswan", tier=tier, m=m)
        for h in [True, False]:
            x = [t["shortcut_pref"] for t in ts if t["query_word_falsified"] == h]
            if len(x) >= 10:
                RES["E1_local_vs_global"][f"{tier}|m{m}|word_falsified={h}"] = boot_ci(x) + (len(x),)
                mu, lo, hi = boot_ci(x)
                print(f"{tier:9s} m={m} query word itself falsified={h!s:5s}: pref={mu:.3f} [{lo:.3f},{hi:.3f}] (n={len(x)})")

### The same results in bits
The model's next-token distribution *is* its prediction (a language model is trained to minimise
-log2 p of the next token, so it is literally a compressor). For each conflict puzzle the generator's
answer is the rule label; we compare the bits the model needs to encode it with an ideal learner that
sees the same demos (noiseless labels: 50% on the rule at m = 0, 100% once the shortcut has a
counterexample) and with uniform guessing (log2 3 bits). Normalized gain
G = (uniform - model) / (uniform - ideal): 0 = no better than guessing, 1 = ideal.

In [ ]:
def bits_summary(ts):
    p = np.array([t["label_probs"] for t in ts])
    i_r = np.array([LABELS.index(t["rule_label"]) for t in ts])
    i_s = np.array([LABELS.index(t["short_label"]) for t in ts])
    p_rule, p_short = p[np.arange(len(ts)), i_r], p[np.arange(len(ts)), i_s]
    m = ts[0]["m"]
    L_model = float(np.mean(-np.log2(np.clip(p_rule, 1e-12, 1))))
    L_ideal = 1.0 if m == 0 else 0.0
    L_unif = float(np.log2(K))
    return dict(bits_model=L_model, bits_ideal=L_ideal, bits_wasted=L_model - L_ideal,
                gain=max(0.0, (L_unif - L_model) / (L_unif - L_ideal)),
                p_rule=float(p_rule.mean()), p_shortcut=float(p_short.mean()),
                p_other=float((1 - p_rule - p_short).mean()), picks_shortcut=float(np.mean(p.argmax(1) == i_s)))

BITS = {f"{tier}|m{m}": bits_summary(by(split="E1_blackswan", tier=tier, m=m)) for tier in TIERS for m in range(K + 1)}
for k, r in BITS.items():
    print(f"{k:12s} model {r['bits_model']:.2f} bits  ideal {r['bits_ideal']:.2f}  wasted {r['bits_wasted']:.2f}  "
          f"gain {r['gain']:.2f} | p(rule) {r['p_rule']:.2f} p(shortcut) {r['p_shortcut']:.2f} p(other) {r['p_other']:.2f}")
RES["E1_bits"] = BITS

fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.8))
ms = list(range(K + 1))
for tier, c in zip(TIERS, ["C0", "C1", "C2"]):
    axs[0].plot(ms, [BITS[f"{tier}|m{m}"]["bits_model"] for m in ms], "o-", color=c, label=f"{MODEL_ID.split('/')[-1]} · {tier}")
    axs[1].plot(ms, [BITS[f"{tier}|m{m}"]["gain"] for m in ms], "o-", color=c, label=tier)
axs[0].plot(ms, [BITS[f"nonsense|m{m}"]["bits_ideal"] for m in ms], "k--", label="ideal learner")
axs[0].axhline(np.log2(K), color="gray", lw=0.8, ls=":", label="uniform guess")
axs[0].set_xlabel("m = counterexamples to the shortcut"); axs[0].set_ylabel("bits to encode the right answer")
axs[0].set_title("Cost in bits (lower is better)"); axs[0].legend(fontsize=7); axs[0].set_xticks(ms)
axs[1].axhline(1, color="k", ls="--", lw=1, label="ideal learner"); axs[1].set_ylim(0, 1.05); axs[1].set_xticks(ms)
axs[1].set_xlabel("m = counterexamples to the shortcut"); axs[1].set_ylabel("normalized gain G")
axs[1].set_title("Share of the learnable structure captured"); axs[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/figures/e1_bits.png", dpi=160); plt.show()

## Experiment 2: count vs fraction
One counterexample, increasingly many confirming demos. An ideal falsifier stays flat. A model that
averages votes lets the counterexample get diluted, so shortcut preference rises with `k_per`.

In [ ]:
E2 = {}
for k_per in [2, 3, 4, 6]:
    ts = by(split="E2_count_vs_fraction", k_per=k_per)
    p = score(ts)["p"]; pr = prefs(ts, p)
    for t, x, pp in zip(ts, pr, p): t["shortcut_pref"] = float(x); t["label_probs"] = pp.tolist()
    E2[k_per] = boot_ci(pr)
    print(f"k_per={k_per} (n_demos={K * k_per:2d}, counterexample fraction={1 / (K * k_per):.3f}): "
          f"pref={E2[k_per][0]:.3f} [{E2[k_per][1]:.3f},{E2[k_per][2]:.3f}]")
RES["E2_count_vs_fraction"] = {str(k): v for k, v in E2.items()}

fig, ax = plt.subplots(figsize=(5.4, 3.8))
kk = sorted(E2); y = [E2[k][0] for k in kk]
ax.errorbar([K * k for k in kk], y, yerr=[[E2[k][0] - E2[k][1] for k in kk], [E2[k][2] - E2[k][0] for k in kk]],
            marker="o", capsize=3, label=f"{MODEL_ID.split('/')[-1]} (m = 1)")
ax.axhline(ideal_pref(1, 0.05), color="k", ls="--", lw=1, label="ideal observer ε=0.05 (flat in n)")
ax.set_xlabel("number of demos (one is a counterexample)"); ax.set_ylabel("shortcut preference")
ax.legend(fontsize=8); ax.set_title("Is one black swan diluted by more white swans?")
fig.tight_layout(); fig.savefig(f"{OUT}/figures/e2_count_vs_fraction.png", dpi=160); plt.show()

## Experiment 3: does the model *know* which feature is the rule?
For tasks with `m >= 1`, the demos determine which slot (first/second word) holds the rule.
We linearly probe that variable from the last-token residual stream at every layer.
**Negative control:** at `m = 0` the demos do *not* determine it (roles are counterbalanced), so a probe must
sit at chance. If it doesn't, the probe is picking up something other than evidence.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

E3 = by(split="E3_internals")
r3 = score(E3, hidden=True, lens=True)
H3, LENS3 = r3["H"], r3["lens"]
pr3 = prefs(E3, r3["p"])
for t, x in zip(E3, pr3): t["shortcut_pref"] = float(x)
m3 = np.array([t["m"] for t in E3]); y_slot = np.array([int(t["rule_first"]) for t in E3])
print("H3", H3.shape)


def probe_curve(X, y, seed=0):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    return [float(cross_val_score(clf, X[:, l], y, cv=cv, scoring="balanced_accuracy").mean()) for l in range(X.shape[1])]


sel = m3 >= 1
curve_evid = probe_curve(H3[sel], y_slot[sel])
curve_null = probe_curve(H3[m3 == 0], y_slot[m3 == 0])
curve_shuf = probe_curve(H3[sel], np.random.default_rng(0).permutation(y_slot[sel]))
behav_acc = float(np.mean(pr3[sel] < 0.5))          # how often the OUTPUT picks the rule
best_l = int(np.argmax(curve_evid))
print(f"probe 'which slot is the rule' (m>=1): best layer {best_l}, bal.acc {curve_evid[best_l]:.3f}")
print(f"  m=0 control best {max(curve_null):.3f} | shuffled-label control best {max(curve_shuf):.3f}")
print(f"  behaviour: output prefers the rule on {behav_acc:.3f} of the same tasks")
RES["E3_probe"] = dict(evidence=curve_evid, m0_control=curve_null, shuffled=curve_shuf,
                       best_layer=best_l, behaviour_rule_rate=behav_acc)

# logit lens: shortcut preference read out at every layer, by m
lens_pref = {m: np.array([prefs([t for t, mm in zip(E3, m3) if mm == m], LENS3[m3 == m][:, l]).mean()
                          for l in range(NL + 1)])
             for m in [0, 1, 2]}
RES["E3_logit_lens"] = {str(m): v.tolist() for m, v in lens_pref.items()}

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
L = np.arange(NL + 1)
axs[0].plot(L, curve_evid, label="rule slot, m≥1 (evidence present)")
axs[0].plot(L, curve_null, label="rule slot, m=0 (no evidence: control)")
axs[0].plot(L, curve_shuf, color="gray", ls=":", label="shuffled labels")
axs[0].axhline(behav_acc, color="C3", ls="--", label=f"behaviour: picks rule {behav_acc:.2f}")
axs[0].set_xlabel("layer"); axs[0].set_ylabel("probe balanced accuracy"); axs[0].legend(fontsize=7)
axs[0].set_title("Knows vs acts")
for m in [0, 1, 2]:
    axs[1].plot(L, lens_pref[m], label=f"m={m}")
axs[1].axhline(0.5, color="gray", lw=0.5); axs[1].set_xlabel("layer"); axs[1].set_ylabel("shortcut pref (logit lens)")
axs[1].legend(fontsize=7); axs[1].set_title("Where the decision forms")
fig.tight_layout(); fig.savefig(f"{OUT}/figures/e3_knows_vs_acts.png", dpi=160); plt.show()

## Experiment 4: causal test — steer along the model's own "rule-slot" direction
Direction `d_l` = mean residual (rule in slot 1) − mean residual (rule in slot 2), from **training** tasks with
`m >= 1`, rescaled to the residual norm. On held-out tasks we add `α · d_l` (last token) signed toward each task's *true* rule slot. If the internal
representation is what the output under-uses, amplifying it should cut shortcut preference. Controls: a random
direction of the same norm, and the anti-direction (should *increase* shortcut preference).
Layer is picked on a validation split and reported on a separate test split.

In [ ]:
idx = np.where(sel)[0]
r = np.random.default_rng(1); r.shuffle(idx)
n_tr = len(idx) // 2; n_va = len(idx) // 4
tr, va, te = idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]

def direction(l):
    """Unit 'rule is in slot 1' direction at layer l, rescaled to the mean residual norm at that layer,
    so alpha is a fraction of the residual stream's size (the raw class-mean difference is tiny)."""
    X, y = H3[tr, l], y_slot[tr]
    d = X[y == 1].mean(0) - X[y == 0].mean(0)
    return d / np.linalg.norm(d) * np.linalg.norm(X, axis=1).mean()

def steer_fn(vec, alpha):
    v = torch.tensor(vec, dtype=dtype, device=DEV)
    def fn(h, bt):
        sgn = torch.tensor([1.0 if t["rule_first"] else -1.0 for t in bt], dtype=dtype, device=DEV)[:, None]
        return h + alpha * sgn * v
    return fn

def run(ids, l, vec, alpha):
    ts = [E3[i] for i in ids]
    return float(prefs(ts, score(ts, hook=(l, steer_fn(vec, alpha)))["p"]).mean())

cand = sorted(set(int(x) for x in np.linspace(4, NL - 2, 8)))
va_res = {l: run(va, l, direction(l), 0.2) for l in cand}
base_va = float(pr3[va].mean())
L_STEER = min(va_res, key=va_res.get)
print("validation (alpha=0.2):", {l: round(v, 3) for l, v in va_res.items()}, "base", round(base_va, 3), "-> layer", L_STEER)

d = direction(L_STEER)
rv = np.random.default_rng(2).standard_normal(d.shape); rv *= np.linalg.norm(d) / np.linalg.norm(rv)
alphas = [0, 0.05, 0.1, 0.2, 0.4]
E4 = {"layer": L_STEER, "alphas": alphas,
      "toward_rule": [run(te, L_STEER, d, a) for a in alphas],
      "anti": [run(te, L_STEER, -d, a) for a in alphas],
      "random": [run(te, L_STEER, rv, a) for a in alphas]}
for k in ["toward_rule", "anti", "random"]:
    print(f"{k:12s}", [round(x, 3) for x in E4[k]])
RES["E4_steering"] = E4

fig, ax = plt.subplots(figsize=(5.4, 3.8))
for k, c in [("toward_rule", "C2"), ("anti", "C3"), ("random", "gray")]:
    ax.plot(alphas, E4[k], marker="o", color=c, label=k)
ax.set_xlabel("α (fraction of mean residual norm)"); ax.set_ylabel("shortcut preference (held-out)")
ax.set_title(f"Steering at layer {L_STEER}"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/figures/e4_steering.png", dpi=160); plt.show()

## Experiment 5 (optional): does thinking mode fix it?
Same tasks in chat form, generated with `enable_thinking=True` vs `False`. Small n (generation is slow on a T4).

In [ ]:
RUN_THINKING = True
N_THINK = 12 if QUICK else 40

def chat_prompt(t, think):
    body = t["prompt"].rsplit("\n", 1)
    msg = ("Here are labelled examples:\n" + body[0] + "\n\nWhat label goes with: " + body[1].replace(" ->", "") +
           " ?\nReply with the label digit only.")
    return tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=think)

@torch.no_grad()
def generate(prompts, max_new):
    outs = []
    for s in range(0, len(prompts), BATCH):
        enc = tok(prompts[s:s + BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def parse(txt, think):
    if think and "</think>" not in txt:
        return "truncated"                          # ran out of tokens mid-thought
    ans = txt.split("</think>")[-1]
    digs = [c for c in ans if c in LABELS]
    return digs[-1] if digs else None

if RUN_THINKING:
    E5 = {}
    for m in [0, 1, 3]:
        ts = by(split="E1_blackswan", tier="nonsense", m=m)[:N_THINK]
        for think, max_new in [(False, 8), (True, 4096)]:
            outs = generate([chat_prompt(t, think) for t in ts], max_new)
            ans = [parse(o, think) for o in outs]
            sc = np.mean([a == t["short_label"] for a, t in zip(ans, ts)])
            ru = np.mean([a == t["rule_label"] for a, t in zip(ans, ts)])
            tr_ = np.mean([a == "truncated" for a in ans])
            E5[f"m{m}|think={think}"] = dict(shortcut=float(sc), rule=float(ru), truncated=float(tr_),
                                             other=float(1 - sc - ru - tr_), outputs=outs)
            print(f"m={m} thinking={think!s:5s}: shortcut {sc:.2f}  rule {ru:.2f}  "
                  f"third label/unparsed {1 - sc - ru - tr_:.2f}  truncated {tr_:.2f}")
    RES["E5_thinking"] = E5

## Save everything

In [ ]:
with open(f"{OUT}/blackswan_eval_scored.jsonl", "w") as f:
    for t in DATA: f.write(json.dumps(t) + "\n")
np.savez_compressed(f"{OUT}/e3_residuals_lasttoken.npz", H=H3.astype(np.float16), m=m3, rule_first=y_slot,
                    shortcut_pref=pr3)
json.dump(RES, open(f"{OUT}/results.json", "w"), indent=2, default=float)
print(sorted(os.listdir(OUT)), sorted(os.listdir(f"{OUT}/figures")))

## Upload to a public Hugging Face Bucket
Create a **write** token at https://huggingface.co/settings/tokens and paste it into the login prompt.
Add this notebook and the README to `fatima_out/` first (Colab: File → Download .ipynb, then upload it in the file pane).

In [ ]:
from huggingface_hub import login, whoami, create_bucket, sync_bucket
login()
BUCKET = f"{whoami()['name']}/blackswan-falsification-eval"
url = create_bucket(BUCKET, private=False, exist_ok=True)
sync_bucket(OUT, f"hf://buckets/{BUCKET}")
print("bucket:", url.url)